# Claude API + Tools

This notebook illustrates how Claude APIs can be used effectively while exploring the functions, parameters and standard usecases.

### Notebook Setup 

In [1]:
import os
import sys
import anthropic
import json

from dotenv import load_dotenv
from IPython.display import Markdown, display

In [2]:
load_dotenv(override=True)
_API_KEY = os.getenv("ANTHROPIC_API_KEY", "").strip()

# Check for the availablility of key
if (not _API_KEY) or (not isinstance(_API_KEY, str)):
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif (not _API_KEY.startswith("sk-ant-")):
    print("An API key was found, but it doesn't start sk-ant-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [3]:
def to_json(response):
    response_dict = response.model_dump()
    json_str = json.dumps(response_dict, indent=2)
    return json_str

In [4]:
client = anthropic.Anthropic(api_key = _API_KEY)
CLAUDE_HAIKU_45_MODEL="claude-haiku-4-5"

### First API query

In [5]:
response = client.messages.create(
    model = CLAUDE_HAIKU_45_MODEL,
    max_tokens = 128,
    messages = [
        {"role": "user", "content": "What is the capital of France?"}
    ]
)

In [6]:
print(to_json(response))

{
  "id": "msg_01TJBWWcg6hoxSXEQvCcGu7p",
  "container": null,
  "content": [
    {
      "citations": null,
      "text": "The capital of France is Paris.",
      "type": "text"
    }
  ],
  "model": "claude-haiku-4-5-20251001",
  "role": "assistant",
  "stop_reason": "end_turn",
  "stop_sequence": null,
  "type": "message",
  "usage": {
    "cache_creation": {
      "ephemeral_1h_input_tokens": 0,
      "ephemeral_5m_input_tokens": 0
    },
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "inference_geo": "not_available",
    "input_tokens": 14,
    "output_tokens": 10,
    "server_tool_use": null,
    "service_tier": "standard"
  }
}


In [7]:
# The actual text
print("Response: ", response.content[0].text)

# Why Claude stopped generating
print("stop reason: ", response.stop_reason)

# Token usage — important for cost tracking
print(f"Input tokens:  {response.usage.input_tokens}")
print(f"Output tokens: {response.usage.output_tokens}")

# Unique ID for this message (useful for logging/debugging)
print(f"Message ID:  {response.id}")

Response:  The capital of France is Paris.
stop reason:  end_turn
Input tokens:  14
Output tokens: 10
Message ID:  msg_01TJBWWcg6hoxSXEQvCcGu7p


`stop_reason` is more important than it looks. It can be:

- `end_turn`: Claude finished naturally
- `max_tokens`: Hit your max_tokens limit — response may be cut off
- `stop_sequence`: Matched one of your custom stop strings
- `tool_use`: Claude wants to call a tool — you need to handle it

### The `system` prompt and `temperature`

In [8]:
def ask(system, question, temperature):
    response = client.messages.create(
        model = CLAUDE_HAIKU_45_MODEL,
        max_tokens = 128,
        temperature = temperature,   # 0.0 = deterministic, 1.0 = creative
        system = system,             # Sets Claude's persona/rules
        messages = [
            {"role": "user", "content": question}
        ]
    )
    return response.content[0].text

question = "Tell me about the sun."

# Precise and factual
print("--- Scientific (temp=0.0) ---")
print(ask(
    system = "You are a precise astrophysicist. Be factual and concise.",
    question = question,
    temperature = 0.0
))

# Creative and expressive
print("\n--- Poetic (temp=1.0) ---")
print(ask(
    system = "You are a poet. Respond only in vivid, lyrical prose.",
    question = question,
    temperature = 1.0
))

--- Scientific (temp=0.0) ---
# The Sun

## Basic Facts
- **Type**: G-type main-sequence star (yellow dwarf)
- **Age**: ~4.6 billion years
- **Mass**: 1.989 × 10³⁰ kg (333,000 Earth masses)
- **Diameter**: 1.39 million km (109 Earth diameters)
- **Distance from Earth**: 149.6 million km (1 AU)

## Structure
- **Core**: ~15 million K; nuclear fusion converts hydrogen to helium
- **

--- Poetic (temp=1.0) ---
# The Luminous Heart

Behold the sun—that ancient furnace burning at the throat of our universe, a god of fire and sustenance who has sung since before the first breath drew meaning from the void. It hangs there, suspended in the velvet darkness, a golden eye watching over all that blooms and breathes beneath its gaze.

What majesty in that fierce orb! A thousand million times the heft of our small Earth, it is a crucible where hydrogen dreams itself into helium, where the very bones of existence are forged in thermonuclear rapture. Each


### Controlling output with `max_tokens` and `stop_sequences`

In [9]:
# --- Example A: max_tokens cuts off the response ---
response_a = client.messages.create(
    model = CLAUDE_HAIKU_45_MODEL,
    max_tokens = 10,              # Very small on purpose
    messages = [{"role": "user", "content": "Write me a short story about a dragon."}]
)

print("=== Example A: max_tokens=10 ===")
print("Text:       ", response_a.content[0].text)
print("stop_reason:", response_a.stop_reason)  # Will be 'max_tokens'!

# --- Example B: stop_sequences halts at a delimiter ---
response_b = client.messages.create(
    model = CLAUDE_HAIKU_45_MODEL,
    max_tokens = 128,
    stop_sequences = ["</answer>"],   # Stop as soon as this string appears
    messages = [{
        "role": "user",
        "content": "What is 2+2? Reply in this format: <answer>NUMBER</answer>"
    }]
)

print("\n=== Example B: stop_sequences=['</answer>'] ===")
print("Text:        ", response_b.content[0].text)
print("stop_reason: ", response_b.stop_reason)    # Will be 'stop_sequence'
print("stop_sequence:", response_b.stop_sequence)  # Shows which one triggered

=== Example A: max_tokens=10 ===
Text:        # The Last Dragon's Choice

The village below
stop_reason: max_tokens

=== Example B: stop_sequences=['</answer>'] ===
Text:         <answer>4
stop_reason:  stop_sequence
stop_sequence: </answer>


### Defining a tool

In [10]:
# A tool is just a JSON schema that describes a function.
# Claude uses this description to decide WHEN and HOW to call it.

tools = [
    {
        "name": "get_weather",
        "description": "Get the current weather for a given city. "
                       "Use this when the user asks about weather anywhere.",
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "The city name, e.g. 'London' or 'Tokyo'"
                },
                "unit": {
                    "type": "string",
                    "enum": ["celsius", "fahrenheit"],
                    "description": "Temperature unit"
                }
            },
            "required": ["city"]   # 'unit' is optional
        }
    }
]

response = client.messages.create(
    model = CLAUDE_HAIKU_45_MODEL,
    max_tokens = 512,
    tools = tools,
    messages = [
        {"role": "user", "content": "What's the weather like in Paris?"}
    ]
)

print("stop_reason:", response.stop_reason)
print("content blocks:", to_json(response.content[0]))

stop_reason: tool_use
content blocks: {
  "id": "toolu_01REym4Bv6VTb77u6e2GyeCU",
  "caller": {
    "type": "direct"
  },
  "input": {
    "city": "Paris"
  },
  "name": "get_weather",
  "type": "tool_use"
}


Notice two things.  
`stop_reason` is `tool_use`, not `end_turn` — this is your signal to handle the tool call before doing anything else.  
The response contains two blocks: an optional text block (Claude's thinking), and a `ToolUseBlock` with the tool name, a unique id, and the input arguments Claude chose.

### Executing the tool and sending the result back

In [11]:
# --- Your real function (could call an API, DB, anything) ---
def get_weather(city: str, unit: str = "celsius") -> dict:
    # Pretend this calls a real weather API
    return {
        "city": city,
        "temperature": 18,
        "unit": unit,
        "condition": "Partly cloudy",
        "humidity": "72%"
    }

tools = [
    {
        "name": "get_weather",
        "description": "Get the current weather for a given city.",
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {"type": "string"},
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}
            },
            "required": ["city"]
        }
    }
]

user_message = "What's the weather like in Paris?"

# ── CALL 1: Claude decides to use the tool ──────────────────────────────────
response = client.messages.create(
    model = CLAUDE_HAIKU_45_MODEL,
    max_tokens = 256,
    tools = tools,
    messages=[{"role": "user", "content": user_message}]
)

print(f"[Call 1] stop_reason: {response.stop_reason}")

# ── Handle the tool call ─────────────────────────────────────────────────────
tool_results = []

for block in response.content:
    if (block.type == "tool_use"):
        print(f"\nClaude wants to call: {block.name}({block.input})")

        # Execute YOUR function with Claude's chosen arguments
        result = get_weather(**block.input)
        print(f"Function returned: {result}")

        tool_results.append({
            "type": "tool_result",
            "tool_use_id": block.id,       # Must match the ToolUseBlock id
            "content": json.dumps(result)  # Always send as a string
        })

# ── CALL 2: Feed result back, Claude gives final answer ─────────────────────
final_response = client.messages.create(
    model = CLAUDE_HAIKU_45_MODEL,
    max_tokens = 256,
    tools = tools,
    messages=[
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": response.content},  # Claude's tool_use block
        {"role": "user", "content": tool_results}            # Your tool result
    ]
)

print(f"\n[Call 2] stop_reason: {final_response.stop_reason}")
print(f"\nFinal answer:\n{final_response.content[0].text}")

[Call 1] stop_reason: tool_use

Claude wants to call: get_weather({'city': 'Paris'})
Function returned: {'city': 'Paris', 'temperature': 18, 'unit': 'celsius', 'condition': 'Partly cloudy', 'humidity': '72%'}

[Call 2] stop_reason: end_turn

Final answer:
The weather in Paris is currently **partly cloudy** with the following conditions:
- **Temperature:** 18°C (about 64°F)
- **Humidity:** 72%

It's a mild day with moderate cloud cover. You might want to bring a light jacket if you're heading out!


### Multiple tools + Claude selection

In [12]:
# --- Define multiple tools ---
tools = [
    {
        "name": "get_weather",
        "description": "Get current weather for a city.",
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {"type": "string"}
            },
            "required": ["city"]
        }
    },
    {
        "name": "calculate",
        "description": "Perform a mathematical calculation. Use for any arithmetic.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "A valid Python math expression, e.g. '17 * 24 + 5'"
                }
            },
            "required": ["expression"]
        }
    },
    {
        "name": "search_database",
        "description": "Search a product database by name.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string"},
                "limit": {"type": "integer", "description": "Max results to return"}
            },
            "required": ["query"]
        }
    }
]

# --- Fake implementations ---
def get_weather(city): return {"city": city, "temp": 22, "condition": "Sunny"}
def calculate(expression): return {"result": eval(expression)}
def search_database(query, limit=3): return {"results": [f"{query} Pro", f"{query} Lite", f"{query} Plus"][:limit]}

TOOL_MAP = {
    "get_weather": get_weather,
    "calculate": calculate,
    "search_database": search_database
}

def run_with_tools(user_question):
    print(f"\nUser: {user_question}")
    messages = [{"role": "user", "content": user_question}]

    response = client.messages.create(
        model = CLAUDE_HAIKU_45_MODEL,
        max_tokens = 256,
        tools = tools,
        messages = messages
    )

    if (response.stop_reason == "tool_use"):
        tool_block = next(b for b in response.content if b.type == "tool_use")
        print(f"  → Claude chose: {tool_block.name}({tool_block.input})")

        fn = TOOL_MAP[tool_block.name]
        result = fn(**tool_block.input)

        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": [
            {"type": "tool_result", "tool_use_id": tool_block.id, "content": json.dumps(result)}
        ]})

        final = client.messages.create(
            model = CLAUDE_HAIKU_45_MODEL,
            max_tokens = 256,
            tools = tools,
            messages = messages
        )
        print(f"  → {final.content[0].text}")
    else:
        print(f"  → {response.content[0].text}")

# Ask three different questions — watch Claude pick the right tool each time
run_with_tools("What's the weather in Tokyo?")
run_with_tools("What is 345 * 78?")
run_with_tools("Search for 'laptop' in the database")


User: What's the weather in Tokyo?
  → Claude chose: get_weather({'city': 'Tokyo'})
  → The weather in Tokyo is currently **sunny** with a temperature of **22°C** (approximately 72°F).

User: What is 345 * 78?
  → Claude chose: calculate({'expression': '345 * 78'})
  → 345 * 78 = **26,910**

User: Search for 'laptop' in the database
  → Claude chose: search_database({'query': 'laptop'})
  → I found 3 laptop products in the database:

1. **laptop Pro**
2. **laptop Lite**
3. **laptop Plus**

Would you like more information about any of these products, or would you like to search for something else?


Claude reads the `description` field of each tool to decide which one to call.  
That `description` is critically important — it's the only thing Claude uses to understand what a tool does.  
Write it like you're explaining the tool to a smart colleague.

### Controlling tool use with `tool_choice`

In [13]:
tools = [
    {
        "name": "get_weather",
        "description": "Get weather for a city.",
        "input_schema": {
            "type": "object",
            "properties": {"city": {"type": "string"}},
            "required": ["city"]
        }
    }
]

msg = [{"role": "user", "content": "Tell me something interesting."}]

# --- tool_choice: "auto" (default) ---
# Claude decides whether to use a tool or answer directly.
r = client.messages.create(
        model = CLAUDE_HAIKU_45_MODEL,
        max_tokens = 256,
        tools = tools,
        tool_choice = {"type": "auto"},
        messages = msg
    )

print("auto →", r.stop_reason, "| used tool?", any(b.type == "tool_use" for b in r.content))
# Output: auto → end_turn | used tool? False  (question didn't need the tool)

# --- tool_choice: "any" ---
# Claude MUST call some tool — it can pick which one.
r = client.messages.create(
    model = CLAUDE_HAIKU_45_MODEL,
    max_tokens = 256,
    tools = tools,
    tool_choice = {"type": "any"},
    messages = msg
)
print("any  →", r.stop_reason, "| tool:", r.content[0].name)
# Output: any → tool_use | tool: get_weather  (forced to pick something)

# --- tool_choice: "tool" (force a specific tool) ---
# Claude MUST call get_weather regardless of the question.
r = client.messages.create(
    model = CLAUDE_HAIKU_45_MODEL,
    max_tokens = 256,
    tools = tools,
    tool_choice = {"type": "tool", "name": "get_weather"},
    messages = msg
)
print("tool →", r.stop_reason, "| tool:", r.content[0].name, "| input:", r.content[0].input)
# Output: tool → tool_use | tool: get_weather | input: {'city': 'London'}  (Claude guesses a city)

# --- tool_choice: "none" ---
# Claude must NOT use any tools, even if it would normally want to.
r = client.messages.create(
    model = CLAUDE_HAIKU_45_MODEL,
    max_tokens = 256,
    tools = tools,
    tool_choice = {"type": "none"},
    messages = [{"role": "user", "content": "What's the weather in Paris?"}]
) 

# Some models return an empty content list when blocked from tools,
# so always guard with a fallback before accessing r.content[0]
text = r.content[0].text[:60] if r.content else "(empty response)"
print("none →", r.stop_reason, "| text:", text, "...")
# Output: none → end_turn | text: I don't have access to real-time weather...

auto → end_turn | used tool? False
any  → tool_use | tool: get_weather
tool → tool_use | tool: get_weather | input: {'city': 'New York'}
none → end_turn | text: (empty response) ...


| tool_choice |                                 When to use it                                |
|:-----------:|:-----------------------------------------------------------------------------:|
| "auto"      | Default. Claude decides. Best for most cases.                                 |
| "any"       | You need Claude to always produce structured data via a tool.                 |
| "tool"      | You want a specific function called every time (e.g. always extract as JSON). |
| "none"      | You've passed tools for context but don't want them called this turn.         |

### Handling tool errors gracefully

In [14]:

tools = [{
    "name": "get_stock_price",
    "description": "Get the current stock price for a ticker symbol.",
    "input_schema": {
        "type": "object",
        "properties": {
            "ticker": {"type": "string", "description": "Stock ticker, e.g. 'AAPL'"}
        },
        "required": ["ticker"]
    }
}]

def get_stock_price(ticker: str):
    # Simulate a failure for unknown tickers
    known = {"AAPL": 189.50, "GOOGL": 141.20, "MSFT": 378.90}
    if ticker not in known:
        raise ValueError(f"Unknown ticker: {ticker}")
    return {"ticker": ticker, "price": known[ticker], "currency": "USD"}

messages = [{"role": "user", "content": "What's the price of ZZZZZ stock?"}]

response = client.messages.create(
    model = CLAUDE_HAIKU_45_MODEL,
    max_tokens = 256,
    tools = tools,
    messages = messages
)

tool_block = next(b for b in response.content if b.type == "tool_use")
print(f"Claude called: {tool_block.name}({tool_block.input})")

messages.append({"role": "assistant", "content": response.content})

# --- Try running the tool; catch errors and report them to Claude ---
try:
    result = get_stock_price(**tool_block.input)
    tool_result_content = json.dumps(result)
    is_error = False
except Exception as e:
    tool_result_content = str(e)   # Send the error message as the result
    is_error = True                # Set is_error=True so Claude knows

print(f"Tool result (is_error={is_error}): {tool_result_content}")

messages.append({"role": "user", "content": [{
    "type": "tool_result",
    "tool_use_id": tool_block.id,
    "content": tool_result_content,
    "is_error": is_error           # Claude uses this to respond appropriately
}]})

final = client.messages.create(
    model = CLAUDE_HAIKU_45_MODEL,
    max_tokens = 256,
    tools = tools,
    messages = messages
)
print(f"\nClaude's response:\n{final.content[0].text}")

Claude called: get_stock_price({'ticker': 'ZZZZZ'})
Tool result (is_error=True): Unknown ticker: ZZZZZ

Claude's response:
I couldn't find a stock with the ticker symbol "ZZZZZ". This appears to be either:

1. **Not a valid ticker symbol** - ZZZZZ doesn't correspond to a real company on the stock market
2. **A typo** - You may have meant a different ticker symbol

Could you please double-check the ticker symbol? If you're looking for a specific company's stock price, feel free to provide the correct ticker and I'll be happy to look it up for you.


When `is_error=True`, Claude understands the tool failed and responds helpfully rather than hallucinating a result.  
Always wrap your tool execution in try/except and use this pattern — it makes your agent far more robust.  